# Notebook 5 — I/O, Errors & Resource Management

**Topics covered:** File Management · CSV & pathlib · Exception Handling · Custom Exceptions · Context Managers

| Topic | Subtopics |
|---|---|
| 1 · File I/O | `open()` modes, reading/writing text files, `pathlib` |
| 2 · CSV Files | `csv.writer` / `csv.DictReader`, working with structured data |
| 3 · Exception Handling | `try/except/else/finally`, exception chaining (`raise ... from`) |
| 4 · Custom Exceptions | Defining exception classes with structured payload |
| 5 · Context Managers | `with` statement, `__enter__`/`__exit__`, timing blocks |

---

### Sample Data
All examples use two consistent person records — run the setup cell below before any other cell.


In [ ]:
# Shared sample data used throughout this notebook
people = [
    {
        "first_name": "Hanu",
        "last_name":  "Madala",
        "dob":        "20 Jan 2002",
        "phone":      "+65 1234 5678",
        "email":      "hanu.m@tdfs.com",
    },
    {
        "first_name": "Shankar",
        "last_name":  "Cherukuri",
        "dob":        "30 Mar 1997",
        "phone":      "98765432",
        "email":      "shankar.c@tdfs.com",
    },
]


---
## Topic 1 — File I/O

Python's built-in `open()` function gives you a file object you can read from or write to. Always specify the mode and encoding. Use `pathlib.Path` for robust, cross-platform path handling.

| Mode | Meaning |
|------|---------|
| `"r"` | Read (default) |
| `"w"` | Write — creates or truncates |
| `"a"` | Append — creates or adds to end |
| `"rb"` / `"wb"` | Binary read / write |


#### 1a · Writing and reading a text file with `open()` and `pathlib`

`pathlib.Path` makes it easy to build paths, check existence, get file metadata, and read/write content without juggling string separators.


In [ ]:
from pathlib import Path

# Build a path in the current working directory
output_path = Path("people.txt")

# --- Write ---
with open(output_path, "w", encoding="utf-8") as f:
    for person in people:
        line = f"{person['first_name']} {person['last_name']} | {person['phone']} | {person['email']}\n"
        f.write(line)

print(f"Written to: {output_path.resolve()}")
print(f"File size : {output_path.stat().st_size} bytes")

# --- Read back ---
print("\n--- File contents ---")
with open(output_path, "r", encoding="utf-8") as f:
    for line in f:
        print(line, end="")


#### 1b · Append mode and line count with `pathlib`

Opening a file in `"a"` mode never truncates existing content — new lines are added after the last byte. `Path.read_text()` reads the entire file as a single string; splitting on `"\n"` gives a quick line count.


In [ ]:
from pathlib import Path

txt_path = Path("people.txt")

# Append a note footer
with open(txt_path, "a", encoding="utf-8") as f:
    f.write("--- end of file ---\n")

# Read entire file and report stats
content = txt_path.read_text(encoding="utf-8")
lines   = [ln for ln in content.splitlines() if ln]   # skip empty lines

print(f"Size      : {txt_path.stat().st_size} bytes")
print(f"Line count: {len(lines)}")
print()
print(content)


---
## Topic 2 — CSV Files

The `csv` module treats each row as a list (with `writer`/`reader`) or as a dict (with `DictWriter`/`DictReader`). `DictWriter` and `DictReader` are the most convenient: column names come from the dict keys, automatically writing/reading headers for you.


#### 2a · Writing a CSV with `csv.DictWriter`

`DictWriter` accepts a list of field names (the header row) and writes each person dict as a data row, aligning values to columns automatically.


In [ ]:
import csv
from pathlib import Path

csv_path = Path("people.csv")
fieldnames = ["first_name", "last_name", "dob", "phone", "email"]

# Write both people to CSV
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()          # writes the header row
    writer.writerows(people)      # writes one row per person

print(f"Created: {csv_path.resolve()}")
print()
print(csv_path.read_text(encoding="utf-8"))


#### 2b · Reading a CSV with `csv.DictReader`

`DictReader` reads the first non-blank row as headers and yields each subsequent row as an `OrderedDict` (or plain `dict` in Python 3.8+), so you can access values by column name.


In [ ]:
import csv
from pathlib import Path

csv_path = Path("people.csv")

print("--- Reading people.csv ---")
with open(csv_path, "r", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        full_name = f"{row['first_name']} {row['last_name']}"
        print(f"{full_name} | {row['phone']} | {row['email']}")


---
## Topic 3 — Exception Handling

Python uses `try / except / else / finally` blocks to intercept and respond to runtime errors. The `else` clause runs only when *no* exception was raised; `finally` always runs, making it ideal for cleanup.

```
try:
    risky_operation()
except SomeError as e:
    handle(e)
else:
    success_path()
finally:
    always_runs()
```


#### 3a · `try / except / else / finally` — reading a JSON file safely

Handling `FileNotFoundError` and `json.JSONDecodeError` separately gives the caller clear, actionable error messages.


In [ ]:
import json
from pathlib import Path

def load_profiles(filepath: str) -> list:
    """Load a JSON file containing a list of person dicts."""
    path = Path(filepath)
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except FileNotFoundError:
        print(f"[ERROR] File not found: {path}")
        return []
    except json.JSONDecodeError as e:
        print(f"[ERROR] Could not parse JSON in {path}: {e}")
        return []
    else:
        # Runs ONLY when no exception was raised
        print(f"[OK] Loaded {len(data)} record(s) from {path}")
        return data
    finally:
        # Always runs — useful for logging or releasing resources
        print(f"[INFO] Attempted to read: {path}")

# Case 1 — file does not exist
load_profiles("profiles.json")

print()

# Case 2 — write a valid JSON file, then load it
json_path = Path("profiles.json")
json_path.write_text(json.dumps(people, indent=2), encoding="utf-8")
result = load_profiles("profiles.json")
for r in result:
    print(f"  {r['first_name']} {r['last_name']}")


#### 3b · Exception chaining — `raise ... from err`

When you catch one exception and raise a different, more specific one, use `raise NewError(...) from original_error`. This preserves the full traceback chain, making it clear what caused what.


In [ ]:
class InvalidPhoneError(ValueError):
    """Raised when a phone string fails validation."""
    def __init__(self, phone: str, reason: str):
        self.phone  = phone
        self.reason = reason
        super().__init__(f"Invalid phone '{phone}': {reason}")

def parse_phone_digits(phone: str) -> int:
    """
    Strip '+', spaces, and dashes then convert to int.
    Uses exception chaining to wrap ValueError from int().
    """
    cleaned = phone.replace("+", "").replace(" ", "").replace("-", "")
    try:
        return int(cleaned)
    except ValueError as err:
        raise InvalidPhoneError(phone, "contains non-digit characters") from err

# Hanu's phone: +65 1234 5678 — strips cleanly to an int
try:
    digits = parse_phone_digits(people[0]["phone"])
    print(f"Hanu digits   : {digits}")
except InvalidPhoneError as e:
    print(f"[FAIL] {e}")

# Simulate a bad phone value to demonstrate chaining
try:
    parse_phone_digits("hello-world")
except InvalidPhoneError as e:
    print(f"\n[FAIL] {e}")
    print(f"Caused by    : {e.__cause__}")


---
## Topic 4 — Custom Exceptions

Python exception classes are ordinary classes that inherit from `Exception` (or a subclass). Adding structured attributes (like `phone` or `email`) lets callers inspect failure details programmatically rather than parsing a string message.


#### 4a · Defining `InvalidPhoneError` and `InvalidEmailError`

Each exception carries structured data. `InvalidPhoneError` stores `phone` and `reason`; `InvalidEmailError` stores the invalid email. Both inherit from `ValueError` so callers can catch either specifically or generically.


In [ ]:
class InvalidPhoneError(ValueError):
    """Raised when a phone string fails validation."""
    def __init__(self, phone: str, reason: str):
        self.phone  = phone
        self.reason = reason
        super().__init__(f"Invalid phone '{phone}': {reason}")

class InvalidEmailError(ValueError):
    """Raised when an email address is structurally invalid."""
    def __init__(self, email: str):
        self.email = email
        super().__init__(f"Invalid email '{email}': missing '@' or unrecognised domain")

# ---- Validation functions ----

KNOWN_DOMAINS = {"tdfs.com"}

def validate_email(email: str) -> None:
    if "@" not in email:
        raise InvalidEmailError(email)
    domain = email.split("@")[-1]
    if domain not in KNOWN_DOMAINS:
        raise InvalidEmailError(email)

def validate_phone(phone: str) -> None:
    stripped = phone.strip()
    if stripped.startswith("+"):
        return   # international format accepted
    digits_only = stripped.replace(" ", "").replace("-", "")
    if not digits_only.isdigit() or len(digits_only) != 8:
        raise InvalidPhoneError(phone, "expected international (+XX ...) or 8-digit local format")

# ---- Test with both persons ----
for person in people:
    try:
        validate_email(person["email"])
        validate_phone(person["phone"])
        print(f"[OK] {person['first_name']} {person['last_name']} — all fields valid")
    except InvalidEmailError as e:
        print(f"[EMAIL ERROR] {e}")
    except InvalidPhoneError as e:
        print(f"[PHONE ERROR] {e}")

# ---- Test with deliberate bad data ----
print()
bad_cases = [
    {"email": "no-at-sign.com", "phone": "+65 1234 5678"},
    {"email": "user@unknown.org", "phone": "98765432"},
    {"email": "valid@tdfs.com",   "phone": "12345"},
]
for case in bad_cases:
    try:
        validate_email(case["email"])
        validate_phone(case["phone"])
    except (InvalidEmailError, InvalidPhoneError) as e:
        print(f"[ERROR] {e}")


#### 4b · `validate_person()` — combining both validators

A single `validate_person()` function orchestrates both checks and collects all errors before raising, giving users one comprehensive report rather than stopping at the first failure.


In [ ]:
def validate_person(record: dict) -> None:
    """
    Run email and phone validation on a person dict.
    Raises the first ValidationError encountered.
    """
    validate_email(record.get("email", ""))
    validate_phone(record.get("phone", ""))

# Valid records
for person in people:
    try:
        validate_person(person)
        print(f"[OK] {person['first_name']} {person['last_name']}")
    except (InvalidEmailError, InvalidPhoneError) as e:
        print(f"[ERROR] {person['first_name']}: {e}")

# Record missing a country code — treated as suspicious
suspect = {
    "first_name": "Test",
    "last_name":  "User",
    "dob":        "01 Jan 1990",
    "phone":      "1234",          # too short, no country code
    "email":      "test@tdfs.com",
}
print()
try:
    validate_person(suspect)
except InvalidPhoneError as e:
    print(f"[PHONE ERROR] {e}")
    print(f"  phone  = {e.phone!r}")
    print(f"  reason = {e.reason!r}")


---
## Topic 5 — Context Managers

The `with` statement calls `__enter__` when the block starts and `__exit__` when it ends, even if an exception is raised. This guarantees cleanup — closing files, releasing locks, stopping timers — without needing `try/finally` everywhere.

```python
with open("file.txt") as f:   # __enter__ returns file object → f
    data = f.read()
# __exit__ closes the file here, even if an exception occurred
```


#### 5a · Class-based context manager — a timing block

Implement `__enter__` to record a start time (and optionally return a value) and `__exit__` to compute elapsed time. This gives any block of code automatic timing without modifying it.


In [ ]:
import time

class Timer:
    """Context manager that measures elapsed wall-clock time."""

    def __enter__(self):
        self._start = time.perf_counter()
        return self                    # available as 'as timer'

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.elapsed_ms = (time.perf_counter() - self._start) * 1000
        print(f"[Timer] elapsed: {self.elapsed_ms:.3f} ms")
        return False                   # do not suppress any exception

from datetime import datetime

def calculate_age(dob_str: str) -> int:
    dob   = datetime.strptime(dob_str, "%d %b %Y")
    today = datetime.today()
    return today.year - dob.year - ((today.month, today.day) < (dob.month, dob.day))

print("--- timing a batch operation ---")
with Timer() as t:
    results = [
        f"{p['first_name']} {p['last_name']}, age {calculate_age(p['dob'])}"
        for p in people
    ]

for r in results:
    print(" ", r)

print(f"\nCaptured elapsed via attribute: {t.elapsed_ms:.3f} ms")


---
## Exercises

Complete each exercise in the cell below it. Reference solutions are hidden in the `<details>` blocks.


### Exercise 1 — Write and read `people.csv`

Write both person records to `people.csv` (columns: `first_name`, `last_name`, `dob`, `phone`, `email`) using `csv.DictWriter`. Then read it back with `csv.DictReader` and print each row as:

```
Hanu Madala | +65 1234 5678 | hanu.m@tdfs.com
Shankar Cherukuri | 98765432 | shankar.c@tdfs.com
```

<details>
<summary>Solution</summary>

```python
import csv
from pathlib import Path

csv_path   = Path("people.csv")
fieldnames = ["first_name", "last_name", "dob", "phone", "email"]

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(people)

with open(csv_path, "r", newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        print(f"{row['first_name']} {row['last_name']} | {row['phone']} | {row['email']}")
```

</details>


In [ ]:
# Exercise 1 — your code here


### Exercise 2 — `validate_person()` with custom exceptions

Define `InvalidPhoneError(phone, reason)` and `InvalidEmailError(email)`. Write `validate_person(record)` that:
- Raises `InvalidPhoneError` if the phone is not international (`+` prefix) and not exactly 8 digits.
- Raises `InvalidEmailError` if `@` is absent.

Test with both person records and confirm only valid ones pass.

<details>
<summary>Solution</summary>

```python
class InvalidPhoneError(ValueError):
    def __init__(self, phone, reason):
        self.phone  = phone
        self.reason = reason
        super().__init__(f"Invalid phone '{phone}': {reason}")

class InvalidEmailError(ValueError):
    def __init__(self, email):
        self.email = email
        super().__init__(f"Invalid email '{email}'")

def validate_person(record):
    email = record.get("email", "")
    if "@" not in email:
        raise InvalidEmailError(email)
    phone   = record.get("phone", "").strip()
    digits  = phone.replace("+", "").replace(" ", "").replace("-", "")
    if not phone.startswith("+") and (not digits.isdigit() or len(digits) != 8):
        raise InvalidPhoneError(phone, "not international and not 8 digits")

for p in people:
    try:
        validate_person(p)
        print(f"[OK] {p['first_name']} {p['last_name']}")
    except (InvalidPhoneError, InvalidEmailError) as e:
        print(f"[ERROR] {e}")
```

</details>


In [ ]:
# Exercise 2 — your code here


### Exercise 3 — Safe JSON loading with `try / except / else / finally`

Simulate a scenario where `profiles.json` may or may not exist:
1. Try to open and parse the file.
2. Catch `FileNotFoundError` and `json.JSONDecodeError` **separately** with distinct messages.
3. In the `else` block, print how many records were loaded.
4. In `finally`, log which file was attempted.

Test with a missing file, then create the file and try again.

<details>
<summary>Solution</summary>

```python
import json
from pathlib import Path

def load_profiles(filepath):
    path = Path(filepath)
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except FileNotFoundError:
        print(f"[ERROR] File not found: {path}")
        return []
    except json.JSONDecodeError as e:
        print(f"[ERROR] Bad JSON in {path}: {e}")
        return []
    else:
        print(f"[OK] Loaded {len(data)} record(s)")
        return data
    finally:
        print(f"[INFO] Attempted: {path}")

load_profiles("profiles.json")      # missing → error
Path("profiles.json").write_text(json.dumps(people, indent=2))
load_profiles("profiles.json")      # exists  → ok
```

</details>


In [ ]:
# Exercise 3 — your code here


### Exercise 4 — Append to CSV and verify row count

Open `people.csv` in **append** mode (`"a"`) and write a third fictional person. Then read the file back with `csv.DictReader` and confirm **three** rows are returned, printing each full name.

<details>
<summary>Solution</summary>

```python
import csv
from pathlib import Path

csv_path = Path("people.csv")
fieldnames = ["first_name", "last_name", "dob", "phone", "email"]

new_person = {
    "first_name": "Alice",
    "last_name":  "Tan",
    "dob":        "15 Jun 1995",
    "phone":      "+65 9876 5432",
    "email":      "alice.t@tdfs.com",
}

with open(csv_path, "a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writerow(new_person)

rows = []
with open(csv_path, "r", newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

print(f"Row count: {len(rows)}")
for row in rows:
    print(f"  {row['first_name']} {row['last_name']}")
```

</details>


In [ ]:
# Exercise 4 — your code here


### Exercise 5 — `pathlib` file inspection

Use `pathlib.Path` to:
1. Check whether `people.csv` exists.
2. If it exists, print its **file size in bytes** and **line count** (including the header).
3. If it does not exist, print a helpful message and create the file.

<details>
<summary>Solution</summary>

```python
from pathlib import Path
import csv

csv_path   = Path("people.csv")
fieldnames = ["first_name", "last_name", "dob", "phone", "email"]

if csv_path.exists():
    size       = csv_path.stat().st_size
    lines      = csv_path.read_text(encoding="utf-8").splitlines()
    line_count = len([l for l in lines if l])
    print(f"File exists: {csv_path.resolve()}")
    print(f"  Size      : {size} bytes")
    print(f"  Line count: {line_count}")
else:
    print(f"{csv_path} does not exist — creating it now.")
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(people)
    print("File created.")
```

</details>


In [ ]:
# Exercise 5 — your code here


### Exercise 6 — Exception chaining with `raise ... from err`

Write a function `parse_phone_digits(phone: str) -> int` that:
1. Strips `+`, spaces, and dashes from the phone string, then calls `int()` on the result.
2. When `int()` raises a `ValueError` (e.g. for Hanu's `+65 1234 5678` which contains spaces), catches it and re-raises as your `InvalidPhoneError` using `raise InvalidPhoneError(...) from err`.

Confirm the `__cause__` chain is visible when you print the exception.

<details>
<summary>Solution</summary>

```python
class InvalidPhoneError(ValueError):
    def __init__(self, phone, reason):
        self.phone  = phone
        self.reason = reason
        super().__init__(f"Invalid phone '{phone}': {reason}")

def parse_phone_digits(phone: str) -> int:
    cleaned = phone.replace("+", "").replace(" ", "").replace("-", "")
    try:
        return int(cleaned)
    except ValueError as err:
        raise InvalidPhoneError(phone, "contains non-digit characters") from err

# Hanu's international number strips cleanly
try:
    print(parse_phone_digits(people[0]["phone"]))
except InvalidPhoneError as e:
    print(f"[FAIL] {e}, caused by: {e.__cause__}")

# Simulate a clearly bad phone
try:
    parse_phone_digits("abc-def")
except InvalidPhoneError as e:
    print(f"[FAIL] {e}")
    print(f"  __cause__: {e.__cause__}")
```

</details>


In [ ]:
# Exercise 6 — your code here


### Exercise 7 — `Timer` context manager

Build a `Timer` class context manager with `__enter__` and `__exit__`. Use it to time a loop that calls `calculate_age()` for both persons **1000 times**. Print the total elapsed time in milliseconds.

<details>
<summary>Solution</summary>

```python
import time
from datetime import datetime

class Timer:
    def __enter__(self):
        self._start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.elapsed_ms = (time.perf_counter() - self._start) * 1000
        print(f"[Timer] {self.elapsed_ms:.3f} ms")
        return False

def calculate_age(dob_str):
    dob   = datetime.strptime(dob_str, "%d %b %Y")
    today = datetime.today()
    return today.year - dob.year - ((today.month, today.day) < (dob.month, dob.day))

with Timer():
    for _ in range(1000):
        for p in people:
            calculate_age(p["dob"])
```

</details>


In [ ]:
# Exercise 7 — your code here
